# QMML - Deep Learning Session 1
## Predict Customer Churn

In this notebook we will:
- Explore the dataset
- Preprocess the data for a neural network
- Build and train a neural network in PyTorch
- Generate a submission

There are **3 tasks** for you to complete. The notebook will not run without them.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
sub   = pd.read_csv('sample_submission.csv')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)

## 2. EDA

In [ ]:
train.head()

In [ ]:
train.dtypes

In [ ]:
train.isnull().sum()

In [ ]:
train.describe()

In [ ]:
# Class balance
churn_counts = train['Churn'].value_counts()
print(churn_counts)
print(f'\nChurn rate: {(train["Churn"] == "Yes").mean():.2%}')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
churn_counts.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'])
ax.set_title('Class Balance')
ax.set_xlabel('Churn')
ax.set_ylabel('Count')
ax.set_xticklabels(['No', 'Yes'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Distributions of numeric features
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, col in enumerate(num_cols):
    train[train['Churn'] == 'No'][col].hist(ax=axes[i], alpha=0.6, bins=30, color='steelblue', label='No')
    train[train['Churn'] == 'Yes'][col].hist(ax=axes[i], alpha=0.6, bins=30, color='tomato', label='Yes')
    axes[i].set_title(col)
    axes[i].legend()

plt.suptitle('Numeric Feature Distributions by Churn')
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by key categorical features
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'gender']

fig, axes = plt.subplots(1, len(cat_cols), figsize=(16, 4))

for i, col in enumerate(cat_cols):
    churn_rate = train.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean()).sort_values(ascending=False)
    churn_rate.plot(kind='bar', ax=axes[i], color='steelblue')
    axes[i].set_title(f'Churn rate by {col}')
    axes[i].set_ylabel('Churn rate')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric features only)
train_numeric = train[num_cols].copy()
train_numeric['TotalCharges'] = pd.to_numeric(train_numeric['TotalCharges'], errors='coerce')
train_numeric['Churn'] = (train['Churn'] == 'Yes').astype(int)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(train_numeric.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
def preprocess(df, is_train=True, encoders=None):
    df = df.copy()

    # Drop id
    df = df.drop(columns=['id'])

    # TotalCharges has some spaces -- coerce to numeric
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(0)

    # Encode target
    if is_train:
        y = (df['Churn'] == 'Yes').astype(int).values
        df = df.drop(columns=['Churn'])
    else:
        y = None

    # Label encode all object columns
    cat_cols = df.select_dtypes(include='object').columns.tolist()

    if is_train:
        encoders = {}
        for col in cat_cols:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
            encoders[col] = le
    else:
        for col in cat_cols:
            df[col] = encoders[col].transform(df[col])

    return df.values, y, encoders


X, y, encoders = preprocess(train, is_train=True)
X_test, _, _   = preprocess(test,  is_train=False, encoders=encoders)

# Scale
scaler = StandardScaler()
X      = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

# Train / val split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', X_train.shape)
print('Val size:  ', X_val.shape)
print('Input features:', X_train.shape[1])

In [ ]:
# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=256,
    shuffle=True
)

INPUT_SIZE = X_train.shape[1]
print('Ready. Input size:', INPUT_SIZE)

## 4. Build the Model

---
### TASK 1 - Define the network architecture

Complete the `__init__` method below. The skeleton has 3 hidden layers already defined for you. Your job is to:

- Choose the size (number of neurons) for each hidden layer
- Choose an activation function to use after each hidden layer

A few things to keep in mind:
- `INPUT_SIZE` is already defined for you (number of input features)
- The output layer is also already written -- it outputs a single value (the churn probability)
- Common activation functions: `nn.ReLU()`, `nn.Tanh()`, `nn.Sigmoid()`
- There is no single correct answer -- experiment!

---

In [ ]:
class ChurnNet(nn.Module):
    def __init__(self, input_size):
        super(ChurnNet, self).__init__()

        # --- TASK 1 START ---
        # Define 3 hidden layers and your chosen activation.
        # Replace each ??? with a value or layer.
        #
        # Example of a single linear layer:
        #   nn.Linear(in_features, out_features)

        self.layer1 = nn.Linear(input_size, ???)
        self.act1   = ???

        self.layer2 = nn.Linear(???, ???)
        self.act2   = ???

        self.layer3 = nn.Linear(???, ???)
        self.act3   = ???

        # --- TASK 1 END ---

        # Output layer -- do not change this
        self.output  = nn.Linear(???, 1)   # final ??? should match layer3 output size
        self.out_act = nn.Sigmoid()

    def forward(self, x):
        x = self.act1(self.layer1(x))
        x = self.act2(self.layer2(x))
        x = self.act3(self.layer3(x))
        x = self.out_act(self.output(x))
        return x

In [ ]:
model = ChurnNet(INPUT_SIZE)
print(model)

## 5. Train the Model

---
### TASK 2 - Write the training loop

The model, loss function, and optimizer are set up for you below. You need to complete the training loop.

For each batch in each epoch, the loop should:
1. Do a **forward pass** to get predictions from the model
2. **Compute the loss** between predictions and true labels
3. **Zero the gradients** (call `optimizer.zero_grad()`)
4. **Backpropagate** the loss (call `.backward()` on the loss)
5. **Update the weights** (call `optimizer.step()`)

---

In [ ]:
LEARNING_RATE = 0.001
EPOCHS        = 20

loss_fn   = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
train_losses = []
val_aucs     = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for X_batch, y_batch in train_loader:

        # --- TASK 2 START ---
        # Write the 5 steps described above.

        pass  # remove this line when you write your code

        # --- TASK 2 END ---

        epoch_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t).squeeze().numpy()
    val_auc = roc_auc_score(y_val, val_preds)

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    val_aucs.append(val_auc)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:>3}/{EPOCHS}  |  Loss: {avg_loss:.4f}  |  Val AUC: {val_auc:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses)
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCE Loss')

ax2.plot(val_aucs, color='tomato')
ax2.set_title('Validation AUC')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('ROC-AUC')

plt.tight_layout()
plt.show()

print(f'Best Val AUC: {max(val_aucs):.4f}')

## 6. Tune the Model

---
### TASK 3 - Improve the validation AUC above 0.85

Go back and experiment with the following:

- **Learning rate** -- try values like `0.01`, `0.0001`
- **Epochs** -- more epochs means more training, but watch for overfitting
- **Hidden layer sizes** -- larger layers can learn more, but take longer
- **Activation functions** -- try swapping `ReLU` for `Tanh` or vice versa
- **Dropout** -- add `nn.Dropout(p=0.3)` after an activation to reduce overfitting

Re-run the training cell each time you make a change. Can you beat **0.85 AUC**?

---

## 7. Generate Submission

In [ ]:
model.eval()
with torch.no_grad():
    test_preds = model(X_test_t).squeeze().numpy()

sub['Churn'] = test_preds
sub.to_csv('submission.csv', index=False)

print('Submission saved.')
sub.head()